# import libraries


In [7]:
import json
import os
import with_llm
import time
import copy
import csv


# install the tools

In [2]:
!pip install ortools

# 1. smoke test

In [5]:
# initial batch of test cases
smoke_data = [
  {
    "test_id": "001",
    "category": "happy_path_with_typos",
    "user_input": "Hanna cannot make it this Sunay evening",
    "expected_parser_output": {
        "type": "unavailable",
        "employee_name": "Hannah",
        "day": "Sunday",
        "shift": "evening"
    }
  },
  {
    "test_id": "002",
    "category": "missing_information",
    "user_input": "Brian wants to swap his shift with George.",
    "expected_parser_output": {
        "error": "Missing day and shift information for the swap."
    }
  },
  {
    "test_id": "003",
    "category": "invalid_employee",
    "user_input": "Please remove David from the Saturday morning schedule.",
    "expected_parser_output": {
        "error": "Employee 'David' not found."
    }
  },
  {
    "test_id": "004",
    "category": "avoid_shift",
    "user_input": "Make sure Fiona never works mornings anymore.",
    "expected_parser_output": {
        "type": "avoid_shift",
        "employee_name": "Fiona",
        "shift": "morning",
        "penalty": 3
    }
  },
  {
    "test_id": "005",
    "category": "avoid_back_to_back_implied",
    "user_input": "Eric is exhausted from working double shifts, please stop doing that to him.",
    "expected_parser_output": {
        "type": "avoid_back_to_back",
        "employee_name": "Eric",
        "penalty": 5
    }
  }
]

# write the data to a JSON file
file_path = "smoke_dataset.json"
with open(file_path, "w", encoding="utf-8") as f:
    json.dump(smoke_data, f, indent=4)

print(f"Successfully created {file_path}!")

Successfully created smoke_dataset.json!


# run smoke test

In [6]:
# 1. Set the API Key
os.environ["GROQ_API_KEY"] = "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"

def evaluate_llm_coordinator(eval_file, data_file):
    print("Starting LLM Coordinator Smoke Test...\n" + "-"*40)

    # Load the test cases
    with open(eval_file, "r", encoding="utf-8") as f:
        test_cases = json.load(f)

    # Load the restaurant data to get staff names
    with open(data_file, "r", encoding="utf-8") as f:
        restaurant_data = json.load(f)
    staff_names = with_llm.get_staff_names(restaurant_data)

    passed = 0
    total = len(test_cases)

    for test in test_cases:
        print(f"Testing ID: {test['test_id']} | Category: {test['category']}")
        print(f"Input: '{test['user_input']}'")

        try:
            # Call the LLM parser
            actual_output = with_llm.parse_user_request(test["user_input"], staff_names)
            expected = test["expected_parser_output"]

            # Basic Exact Match Evaluation [cite: 59]
            if actual_output == expected:
                print("✅ PASS: Output matched exactly.")
                passed += 1
            else:
                print("❌ FAIL: Output mismatch.")
                print(f"   Expected: {expected}")
                print(f"   Actual:   {actual_output}")

        except Exception as e:
            # Handle cases where the LLM returns an error (like our missing_info test case)
            expected = test["expected_parser_output"]
            if "error" in expected:
                print("✅ PASS: Expected error caught correctly.")
                passed += 1
            else:
                print(f"❌ FAIL: Unexpected Exception -> {e}")

        print("-" * 40)

    # Calculate final accuracy score
    accuracy = (passed / total) * 100
    print(f"\nSmoke Test Complete! Accuracy: {accuracy}% ({passed}/{total} passed)")

# Run the evaluation
evaluate_llm_coordinator("smoke_dataset.json", "restaurant_data.json")

Starting LLM Coordinator Smoke Test...
----------------------------------------
Testing ID: 001 | Category: happy_path_with_typos
Input: 'Hanna cannot make it this Sunay evening'
✅ PASS: Output matched exactly.
----------------------------------------
Testing ID: 002 | Category: missing_information
Input: 'Brian wants to swap his shift with George.'
✅ PASS: Expected error caught correctly.
----------------------------------------
Testing ID: 003 | Category: invalid_employee
Input: 'Please remove David from the Saturday morning schedule.'
✅ PASS: Expected error caught correctly.
----------------------------------------
Testing ID: 004 | Category: avoid_shift
Input: 'Make sure Fiona never works mornings anymore.'
✅ PASS: Output matched exactly.
----------------------------------------
Testing ID: 005 | Category: avoid_back_to_back_implied
Input: 'Eric is exhausted from working double shifts, please stop doing that to him.'
✅ PASS: Output matched exactly.
---------------------------------

# 2. formal test cases for coordinator

In [3]:
full_eval_data = [
  {"test_id": "001", "category": "happy_path_with_typos", "user_input": "Hanna cannot make it this Sunay evening", "expected_parser_output": {"type": "unavailable", "employee_name": "Hannah", "day": "Sunday", "shift": "evening"}},
  {"test_id": "002", "category": "missing_information", "user_input": "Brian wants to swap his shift with George.", "expected_parser_output": {"error": "Missing day and shift information for the swap."}},
  {"test_id": "003", "category": "invalid_employee", "user_input": "Please remove David from the Saturday morning schedule.", "expected_parser_output": {"error": "Employee 'David' not found."}},
  {"test_id": "004", "category": "avoid_shift", "user_input": "Make sure Fiona never works mornings anymore.", "expected_parser_output": {"type": "avoid_shift", "employee_name": "Fiona", "shift": "morning", "penalty": 3}},
  {"test_id": "005", "category": "avoid_back_to_back_implied", "user_input": "Eric is exhausted from working double shifts, please stop doing that to him.", "expected_parser_output": {"type": "avoid_back_to_back", "employee_name": "Eric", "penalty": 5}},
  {"test_id": "006", "category": "core_unavailable", "user_input": "Alice is sick and cannot work Tuesday morning.", "expected_parser_output": {"type": "unavailable", "employee_name": "Alice", "day": "Tuesday", "shift": "morning"}},
  {"test_id": "007", "category": "core_direct_swap", "user_input": "Can Cindy swap her Friday evening shift with Julia's Saturday morning?", "expected_parser_output": {"type": "direct_swap", "employee_name_1": "Cindy", "day_1": "Friday", "shift_1": "evening", "employee_name_2": "Julia", "day_2": "Saturday", "shift_2": "morning"}},
  {"test_id": "008", "category": "core_avoid_shift_day", "user_input": "Please don't schedule George on Sundays.", "expected_parser_output": {"type": "avoid_shift", "employee_name": "George", "day": "Sunday", "penalty": 3}},
  {"test_id": "009", "category": "core_avoid_shift_time", "user_input": "Avoid giving Daniel evening shifts.", "expected_parser_output": {"type": "avoid_shift", "employee_name": "Daniel", "shift": "evening", "penalty": 3}},
  {"test_id": "010", "category": "lexical_typos_and_slang", "user_input": "Yeah so um Ian is out for Mon eve.", "expected_parser_output": {"type": "unavailable", "employee_name": "Ian", "day": "Monday", "shift": "evening"}},
  {"test_id": "011", "category": "lexical_typos_and_slang", "user_input": "Pls keep brian off the sched on weds.", "expected_parser_output": {"type": "avoid_shift", "employee_name": "Brian", "day": "Wednesday", "penalty": 3}},
  {"test_id": "012", "category": "lexical_typos_and_slang", "user_input": "Swap Eric's Wednesday evening with Alice's Thursday morning.", "expected_parser_output": {"type": "direct_swap", "employee_name_1": "Eric", "day_1": "Wednesday", "shift_1": "evening", "employee_name_2": "Alice", "day_2": "Thursday", "shift_2": "morning"}},
  {"test_id": "013", "category": "incomplete_missing_day", "user_input": "Take Fiona off the morning shift.", "expected_parser_output": {"error": "Missing day information"}},
  {"test_id": "014", "category": "incomplete_missing_swap_target", "user_input": "Swap Ian's Monday morning with Brian.", "expected_parser_output": {"error": "Missing target shift for Brian"}},
  {"test_id": "015", "category": "impossible_invalid_employee", "user_input": "Give back-to-back protection to Zach.", "expected_parser_output": {"error": "Employee 'Zach' not found"}},
  {"test_id": "016", "category": "impossible_invalid_shift", "user_input": "Alice can't work the afternoon shift on Friday.", "expected_parser_output": {"type": "unavailable", "employee_name": "Alice", "day": "Friday", "shift": "evening"}},
  {"test_id": "017", "category": "complex_implied_intent", "user_input": "Eric needs a break between shifts, no doubles.", "expected_parser_output": {"type": "avoid_back_to_back", "employee_name": "Eric", "penalty": 5}},
  {"test_id": "018", "category": "complex_implied_intent", "user_input": "George has a doctor's appointment next Tuesday at 9 AM, so he won't be in.", "expected_parser_output": {"type": "unavailable", "employee_name": "George", "day": "Tuesday", "shift": "morning"}},
  {"test_id": "019", "category": "complex_implied_intent", "user_input": "I prefer not to have Cindy working the Saturday evening rush.", "expected_parser_output": {"type": "avoid_shift", "employee_name": "Cindy", "day": "Saturday", "shift": "evening", "penalty": 3}},
  {"test_id": "020", "category": "core_avoid_back_to_back", "user_input": "Prevent Hannah from doing clopens (morning and evening on the same day).", "expected_parser_output": {"type": "avoid_back_to_back", "employee_name": "Hannah", "penalty": 5}}
]

with open("eval_dataset.json", "w", encoding="utf-8") as f:
    json.dump(full_eval_data, f, indent=4)

print("Successfully rebuilt eval_dataset.json with all 20 cases!")

Successfully rebuilt eval_dataset.json with all 20 cases!


# run the formal test

In [3]:
# 1. Set the API Key
os.environ["GROQ_API_KEY"] = "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"

def evaluate_llm_coordinator(eval_file, data_file):
    print("Starting LLM Coordinator Formal Test...\n" + "-"*40)

    with open(eval_file, "r", encoding="utf-8") as f:
        test_cases = json.load(f)

    with open(data_file, "r", encoding="utf-8") as f:
        restaurant_data = json.load(f)
    staff_names = with_llm.get_staff_names(restaurant_data)

    passed = 0
    total = len(test_cases)

    for test in test_cases:
        print(f"Testing ID: {test['test_id']} | Category: {test['category']}")
        print(f"Input: '{test['user_input']}'")

        try:
            actual_output = with_llm.parse_user_request(test["user_input"], staff_names)
            expected = test["expected_parser_output"]

            if actual_output == expected:
                print("✅ PASS: Output matched exactly.")
                passed += 1
            else:
                print("❌ FAIL: Output mismatch.")
                print(f"   Expected: {expected}")
                print(f"   Actual:   {actual_output}")

        except Exception as e:
            expected = test["expected_parser_output"]
            if "error" in expected:
                print("✅ PASS: Expected error caught correctly.")
                passed += 1
            else:
                print(f"❌ FAIL: Unexpected Exception -> {e}")

        print("-" * 40)

        # Pause for 3 seconds to avoid Groq's 429 Rate Limit
        time.sleep(3)

    accuracy = (passed / total) * 100
    print(f"\nTest Complete! Accuracy: {accuracy}% ({passed}/{total} passed)")

evaluate_llm_coordinator("eval_dataset.json", "restaurant_data.json")

Starting LLM Coordinator Formal Test...
----------------------------------------
Testing ID: 001 | Category: happy_path_with_typos
Input: 'Hanna cannot make it this Sunay evening'
✅ PASS: Output matched exactly.
----------------------------------------
Testing ID: 002 | Category: missing_information
Input: 'Brian wants to swap his shift with George.'
✅ PASS: Expected error caught correctly.
----------------------------------------
Testing ID: 003 | Category: invalid_employee
Input: 'Please remove David from the Saturday morning schedule.'
✅ PASS: Expected error caught correctly.
----------------------------------------
Testing ID: 004 | Category: avoid_shift
Input: 'Make sure Fiona never works mornings anymore.'
✅ PASS: Output matched exactly.
----------------------------------------
Testing ID: 005 | Category: avoid_back_to_back_implied
Input: 'Eric is exhausted from working double shifts, please stop doing that to him.'
✅ PASS: Output matched exactly.
--------------------------------

# 3. test the solver

In [9]:
def evaluate_solver(data_file):
    print("Starting Scheduling Engine (Solver) Evaluation...\n" + "-"*50)

    try:
        data = with_llm.load_data_from_json(data_file)
        initial_schedule = with_llm.generate_schedule(data)
    except Exception as e:
        print(f"Failed to load baseline: {e}")
        return

    # Expanded 10-Case Dataset
    solver_tests = [
        # --- HAPPY PATHS (Core functionality) ---
        {
            "test_id": "S001", "name": "Unavailable Intent",
            "change_request": {"type": "unavailable", "employee_name": "Alice", "day": "Friday", "shift": "morning"},
            "validation_rule": "check_absence"
        },
        {
            "test_id": "S002", "name": "Direct Swap Intent",
            "change_request": {"type": "direct_swap", "employee_name_1": "Brian", "day_1": "Monday", "shift_1": "evening", "employee_name_2": "Ian", "day_2": "Tuesday", "shift_2": "evening"},
            "validation_rule": "check_swap"
        },
        {
            "test_id": "S003", "name": "Avoid Back-to-Back (Soft Constraint)",
            "change_request": {"type": "avoid_back_to_back", "employee_name": "Eric", "penalty": 8},
            "validation_rule": "check_reduction", "target": "back_to_back"
        },
        {
            "test_id": "S004", "name": "Avoid Shift Time (Soft Constraint)",
            "change_request": {"type": "avoid_shift", "employee_name": "Alice", "shift": "morning", "penalty": 3},
            "validation_rule": "check_reduction", "target": "specific_shift"
        },

        # --- EDGE CASES & EXPECTED FAILURES (The Hard Wall) ---
        {
            "test_id": "S005", "name": "Fail: Skill Mismatch on Swap",
            # Fiona is Cashier-only, Alice is Server-only. They cannot mathematically swap.
            "change_request": {"type": "direct_swap", "employee_name_1": "Fiona", "day_1": "Monday", "shift_1": "morning", "employee_name_2": "Alice", "day_2": "Friday", "shift_2": "morning"},
            "validation_rule": "expect_error"
        },
        {
            "test_id": "S006", "name": "Fail: Swap Phantom Shift",
            # Alice is not scheduled for Monday evening.
            "change_request": {"type": "direct_swap", "employee_name_1": "Alice", "day_1": "Monday", "shift_1": "evening", "employee_name_2": "Eric", "day_2": "Monday", "shift_2": "evening"},
            "validation_rule": "expect_error"
        },
        {
            "test_id": "S007", "name": "Fail: Swap Self",
            # Eric cannot swap with himself.
            "change_request": {"type": "direct_swap", "employee_name_1": "Eric", "day_1": "Monday", "shift_1": "evening", "employee_name_2": "Eric", "day_2": "Wednesday", "shift_2": "evening"},
            "validation_rule": "expect_error"
        },
        {
            "test_id": "S008", "name": "Fail: Invalid System Request",
            # Sending a garbage intent type that bypassed the LLM somehow
            "change_request": {"type": "fire_employee", "employee_name": "Alice"},
            "validation_rule": "expect_error"
        },

        # --- MORE SOFT CONSTRAINTS ---
        {
            "test_id": "S009", "name": "Avoid Specific Day (Soft Constraint)",
            "change_request": {"type": "avoid_shift", "employee_name": "George", "day": "Saturday", "penalty": 5},
            "validation_rule": "check_reduction", "target": "specific_shift"
        },
        {
            "test_id": "S010", "name": "Avoid Day/Shift Combo (Soft Constraint)",
            "change_request": {"type": "avoid_shift", "employee_name": "Brian", "day": "Saturday", "shift": "evening", "penalty": 2},
            "validation_rule": "check_reduction", "target": "specific_shift"
        }
    ]

    passed = 0

    for test in solver_tests:
        print(f"Testing ID: {test['test_id']} | Action: {test['name']}")
        change = test["change_request"]

        try:
            test_data = copy.deepcopy(data)
            updated_schedule, updated_data = with_llm.update_schedule(test_data, initial_schedule, change)

            # --- VALIDATION RULES ---
            if test["validation_rule"] == "expect_error":
                print("❌ FAIL: Solver executed an impossible request! It should have crashed.")

            elif test["validation_rule"] == "check_absence":
                violation = [row for row in updated_schedule if row["employee_name"].lower() == change["employee_name"].lower() and row["day"] == change["day"] and row["shift"] == change["shift"]]
                if len(violation) == 0:
                    print("✅ PASS: Employee successfully removed.")
                    passed += 1
                else: print("❌ FAIL: Employee still found in shift.")

            elif test["validation_rule"] == "check_swap":
                new_roles_1 = with_llm.get_employee_assignment_roles(updated_schedule, change["employee_name_1"], change["day_2"], change["shift_2"])
                new_roles_2 = with_llm.get_employee_assignment_roles(updated_schedule, change["employee_name_2"], change["day_1"], change["shift_1"])
                if len(new_roles_1) > 0 and len(new_roles_2) > 0:
                    print("✅ PASS: Direct swap successfully executed.")
                    passed += 1
                else: print("❌ FAIL: Swap targets not reassigned.")

            elif test["validation_rule"] == "check_reduction":
                target_name = change["employee_name"].lower()

                def count_instances(schedule):
                    count = 0
                    if test["target"] == "specific_shift":
                        for row in schedule:
                            day_match = True if "day" not in change else row["day"] == change["day"]
                            shift_match = True if "shift" not in change else row["shift"] == change["shift"]
                            if row["employee_name"].lower() == target_name and day_match and shift_match:
                                count += 1
                    elif test["target"] == "back_to_back":
                        days = {row["day"] for row in schedule}
                        for day in days:
                            has_morning = any(r["employee_name"].lower() == target_name and r["day"] == day and r["shift"] == "morning" for r in schedule)
                            has_evening = any(r["employee_name"].lower() == target_name and r["day"] == day and r["shift"] == "evening" for r in schedule)
                            if has_morning and has_evening:
                                count += 1
                    return count

                initial_count = count_instances(initial_schedule)
                updated_count = count_instances(updated_schedule)

                print(f"   Metric: Instances reduced from {initial_count} to {updated_count}")
                if updated_count <= initial_count:
                    print("✅ PASS: Optimization objective met or maintained.")
                    passed += 1
                else:
                    print("❌ FAIL: Soft constraint increased unexpectedly.")

        except Exception as e:
            # --- HANDLE EXPECTED CRASHES ---
            if test["validation_rule"] == "expect_error":
                print(f"✅ PASS: Expected solver rejection caught -> {e}")
                passed += 1
            else:
                print(f"❌ FAIL: Solver crashed unexpectedly -> {e}")

        print("-" * 50)

    print(f"\nSolver Test Complete! Score: {passed}/{len(solver_tests)} passed")

evaluate_solver("restaurant_data.json")

Starting Scheduling Engine (Solver) Evaluation...
--------------------------------------------------
Testing ID: S001 | Action: Unavailable Intent
✅ PASS: Employee successfully removed.
--------------------------------------------------
Testing ID: S002 | Action: Direct Swap Intent
✅ PASS: Direct swap successfully executed.
--------------------------------------------------
Testing ID: S003 | Action: Avoid Back-to-Back (Soft Constraint)
   Metric: Instances reduced from 1 to 1
✅ PASS: Optimization objective met or maintained.
--------------------------------------------------
Testing ID: S004 | Action: Avoid Shift Time (Soft Constraint)
   Metric: Instances reduced from 1 to 1
✅ PASS: Optimization objective met or maintained.
--------------------------------------------------
Testing ID: S005 | Action: Fail: Skill Mismatch on Swap
✅ PASS: Expected solver rejection caught -> Fiona lacks skill 'server' needed for the swap.
--------------------------------------------------
Testing ID: S0

# 4. test the explainer

In [8]:
def build_explainer_eval_sheet(data_file, output_csv="hitl_eval_sheet.csv"):
    print("Building HITL Evaluation Sheet for the Explainer...\n" + "-"*50)

    try:
        data = with_llm.load_data_from_json(data_file)
        initial_schedule = with_llm.generate_schedule(data)
    except Exception as e:
        print(f"Failed to load baseline: {e}")
        return

    # We will use a few mock requests to generate the initial sheet
    explainer_tests = [
        {"type": "unavailable", "employee_name": "Alice", "day": "Friday", "shift": "morning"},
        {"type": "direct_swap", "employee_name_1": "Brian", "day_1": "Monday", "shift_1": "evening", "employee_name_2": "Ian", "day_2": "Tuesday", "shift_2": "evening"},
        {"type": "avoid_back_to_back", "employee_name": "Eric", "penalty": 8}
    ]

    results = []

    for change in explainer_tests:
        test_data = copy.deepcopy(data)

        try:
            # 1. Run the solver
            updated_schedule, updated_data = with_llm.update_schedule(test_data, initial_schedule, change)

            # 2. Generate the explanation
            explanation = with_llm.generate_explanation(data, initial_schedule, updated_schedule, change)

            # 3. Format the row for our grading rubric
            results.append({
                "Intent Type": change["type"],
                "JSON Request": json.dumps(change),
                "System Explanation": explanation,
                "Human Score (0-5)": "",  # Blank for you/your teammates to fill out
                "Reviewer Notes": ""      # Blank for context
            })
            print(f"✅ Explanation successfully captured for: {change['type']}")

        except Exception as e:
            print(f"❌ Failed to process {change['type']} -> {e}")

    # 4. Export to CSV
    with open(output_csv, mode="w", newline="", encoding="utf-8") as f:
        fieldnames = ["Intent Type", "JSON Request", "System Explanation", "Human Score (0-5)", "Reviewer Notes"]
        writer = csv.DictWriter(f, fieldnames=fieldnames)

        writer.writeheader()
        writer.writerows(results)

    print("-" * 50)
    print(f"Successfully generated {output_csv}!")

# Run it!
build_explainer_eval_sheet("restaurant_data.json")

Building HITL Evaluation Sheet for the Explainer...
--------------------------------------------------
✅ Explanation successfully captured for: unavailable
✅ Explanation successfully captured for: direct_swap
✅ Explanation successfully captured for: avoid_back_to_back
--------------------------------------------------
Successfully generated hitl_eval_sheet.csv!
